# 多 Agent 协作

多 Agent 系统将复杂任务拆分给多个 Specialist Agent（专业 Agent）协作完成。

In [1]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

环境变量已加载


## 为什么需要多 Agent

单一 Agent 拥有的工具越多，越容易混淆。多 Agent 模式通过分工降低复杂度。

## 创建多 Agent 系统

1. 创建多个 Specialist Agent
2. 创建一个 Supervisor Agent 协调分工

In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool

# Agent 1：课程顾问
@tool
def search_course(keyword: str) -> str:
    """搜索菜鸟教程课程"""
    courses = {"python": "Python3 基础教程（免费）", "html": "HTML 基础教程（免费）"}
    return courses.get(keyword.lower(), f"未找到 {keyword}")

course_advisor = create_agent(
    model=init_chat_model("deepseek:deepseek-v4-flash", temperature=0),
    tools=[search_course],
    system_prompt="你是课程顾问，只负责推荐课程。",
    name="course_advisor",
)

# Agent 2：学习规划师
@tool
def generate_plan(goal: str) -> str:
    """生成学习计划"""
    return f"针对「{goal}」的学习计划：每周 3 章，预计 6 周完成"

study_planner = create_agent(
    model=init_chat_model("deepseek:deepseek-v4-flash", temperature=0),
    tools=[generate_plan],
    system_prompt="你是学习规划师，负责制定学习计划。",
    name="study_planner",
)

print(f"已创建 2 个 Specialist Agent: {course_advisor.name}, {study_planner.name}")

已创建 2 个 Specialist Agent: course_advisor, study_planner


## Supervisor Agent（协调者）

Supervisor 接收用户请求，判断由哪个 Specialist Agent 处理，或组合多个 Agent 的结果。

In [4]:
from langchain.agents import create_agent

supervisor = create_agent(
    model=init_chat_model("deepseek:deepseek-v4-flash", temperature=0),
    tools=[],  # Supervisor 本身不执行工具，只做规划
    system_prompt="你是协调者，负责将用户需求分配给最合适的专业助手。",
    name="supervisor",
)

result = supervisor.invoke({"messages": [HumanMessage(content="我想学 Python，有什么课程？学完需要多久？")]})
print(result["messages"][-1].content)
print("Supervisor Agent 已创建")

您的问题是关于学习 Python 的课程推荐和时间规划，最适合的专业助手是 **教育顾问**（或 **Python 学习导师**）。我会将您的需求分配给他，他可以根据您的学习目标（如零基础入门、就业转行、数据分析等）和每天可投入时间，为您推荐具体的课程（免费/付费、在线/书籍）并给出合理的学习周期预估。

请稍等，教育顾问将为您详细解答。
Supervisor Agent 已创建


**术语：**

- **Specialist Agent**：专注于单一领域的专业 Agent
- **Supervisor Agent**：协调者，分配任务给 Specialist
- **多 Agent 协作**：多个 Agent 分工完成复杂任务